In [1]:
from pathlib import Path
import re
import json
from pprint import pprint

# -------------------------------------------------------------------
# CORPUS DIRECTORY
# -------------------------------------------------------------------

CORPUS_DIR = Path("./corpus/zoning")

print("Corpus directory:", CORPUS_DIR.resolve())

Corpus directory: D:\sk\planso_assignment\corpus\zoning


In [2]:
# -------------------------------------------------------------------
# LOAD ALL MARKDOWN FILES
# -------------------------------------------------------------------

markdown_file_paths = sorted(CORPUS_DIR.glob("*.md"))

raw_markdown_files = []

for file_path in markdown_file_paths:
    
    file_text = file_path.read_text(
        encoding="utf-8"
    )
    
    raw_markdown_files.append({
        "source_file": file_path.name,
        "file_path": str(file_path),
        "text": file_text
    })

print(f"Loaded markdown files: {len(raw_markdown_files)}")

Loaded markdown files: 10


In [3]:
# -------------------------------------------------------------------
# VERIFY LOADED FILES
# -------------------------------------------------------------------

print("FILES FOUND:\n")

for idx, file_record in enumerate(raw_markdown_files, start=1):
    
    print(f"{idx:02d}. {file_record['source_file']}")

FILES FOUND:

01. zr_01_rules_of_construction.md
02. zr_02_definitions_key.md
03. zr_03_rear_yard_requirements.md
04. zr_04_permitted_obstructions_rear_yard.md
05. zr_05_floor_area_R6_R12_current.md
06. zr_06_floor_area_R6_R10_SUPERSEDED_2019.md
07. zr_07_front_yard_requirements.md
08. zr_08_permitted_obstructions_all_yards.md
09. zr_09_ceqr_e_designations.md
10. zr_10_height_setback_R6_R12.md


In [4]:
# -------------------------------------------------------------------
# REGEX PATTERNS
# -------------------------------------------------------------------

SECTION_HEADER_PATTERN = re.compile(
    r"^##\s+Section\s+([^\n:]+):\s*(.+)$",
    re.MULTILINE
)

SUBSECTION_HEADER_PATTERN = re.compile(
    r"^###\s+([0-9A-Za-z\-\(\)]+):\s*(.+)$",
    re.MULTILINE
)


# -------------------------------------------------------------------
# PARSE SINGLE MARKDOWN FILE
# -------------------------------------------------------------------

def parse_markdown_file(file_record):
    
    source_file = file_record["source_file"]
    text = file_record["text"]
    
    parsed_sections = []
    
    # ---------------------------------------------------------------
    # FIND MAIN SECTION
    # ---------------------------------------------------------------
    
    main_section_match = SECTION_HEADER_PATTERN.search(text)
    
    if main_section_match:
        
        parent_section_id = main_section_match.group(1).strip()
        parent_section_title = main_section_match.group(2).strip()
        
    else:
        
        parent_section_id = "UNKNOWN"
        parent_section_title = "UNKNOWN"
    
    # ---------------------------------------------------------------
    # FIND SUBSECTIONS
    # ---------------------------------------------------------------
    
    subsection_matches = list(
        SUBSECTION_HEADER_PATTERN.finditer(text)
    )
    
    # ---------------------------------------------------------------
    # CASE 1:
    # FILE HAS SUBSECTIONS
    # ---------------------------------------------------------------
    
    if subsection_matches:
        
        for idx, match in enumerate(subsection_matches):
            
            subsection_id = match.group(1).strip()
            subsection_title = match.group(2).strip()
            
            start_pos = match.end()
            
            if idx < len(subsection_matches) - 1:
                end_pos = subsection_matches[idx + 1].start()
            else:
                end_pos = len(text)
            
            subsection_content = text[start_pos:end_pos].strip()
            
            parsed_sections.append({
                "source_file": source_file,
                
                "parent_section_id": parent_section_id,
                "parent_section_title": parent_section_title,
                
                "section_id": subsection_id,
                "section_title": subsection_title,
                
                "content": subsection_content
            })
    
    # ---------------------------------------------------------------
    # CASE 2:
    # FILE HAS NO SUBSECTIONS
    # ENTIRE SECTION BECOMES SINGLE LEGAL UNIT
    # ---------------------------------------------------------------
    
    else:
        
        parsed_sections.append({
            "source_file": source_file,
            
            "parent_section_id": parent_section_id,
            "parent_section_title": parent_section_title,
            
            "section_id": parent_section_id,
            "section_title": parent_section_title,
            
            "content": text.strip()
        })
    
    return parsed_sections

In [5]:
# -------------------------------------------------------------------
# TEST PARSER ON ONE FILE
# -------------------------------------------------------------------

sample_file = raw_markdown_files[2]

print("SOURCE FILE:")
print(sample_file["source_file"])

print("\n" + "="*80 + "\n")

sample_parsed_sections = parse_markdown_file(sample_file)

print(f"Parsed sections: {len(sample_parsed_sections)}\n")

for section in sample_parsed_sections:
    
    print("SECTION ID:", section["section_id"])
    print("SECTION TITLE:", section["section_title"])
    
    print("\nCONTENT PREVIEW:")
    print(section["content"][:250])
    
    print("\n" + "-"*80 + "\n")

SOURCE FILE:
zr_03_rear_yard_requirements.md


Parsed sections: 3

SECTION ID: 23-342
SECTION TITLE: Rear Yard Requirements

CONTENT PREVIEW:
**Applicable Districts:** R1 through R10

In all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required r

--------------------------------------------------------------------------------

SECTION ID: 23-343
SECTION TITLE: Rear Yard Equivalent Requirements

CONTENT PREVIEW:
Where a **zoning lot** does not have a **rear lot line** as defined in this Resolution, a **rear yard equivalent** shall be provided. The **rear yard equivalent** shall comply with the same minimum depth requirements as specified in Section 23-342.



--------------------------------------------------------------------------------

SECTION ID: 23-344
SECTION TITLE: Additional Rear Yard Modifications

CONTENT PREVIEW:
The **rear yard**

In [6]:
# -------------------------------------------------------------------
# PARSE ENTIRE CORPUS
# -------------------------------------------------------------------

all_parsed_sections = []

for file_record in raw_markdown_files:
    
    parsed_sections = parse_markdown_file(file_record)
    
    all_parsed_sections.extend(parsed_sections)

print("TOTAL PARSED LEGAL SECTIONS:")
print(len(all_parsed_sections))

TOTAL PARSED LEGAL SECTIONS:
15


In [7]:
# -------------------------------------------------------------------
# INSPECT PARSED SECTIONS
# -------------------------------------------------------------------

for idx, section in enumerate(all_parsed_sections, start=1):
    
    print("=" * 100)
    
    print(f"SECTION #{idx}")
    
    print(f"SOURCE FILE      : {section['source_file']}")
    print(f"PARENT SECTION   : {section['parent_section_id']}")
    print(f"SECTION ID       : {section['section_id']}")
    print(f"SECTION TITLE    : {section['section_title']}")
    
    content_preview = section["content"][:300]
    
    print("\nCONTENT PREVIEW:")
    print(content_preview)
    
    print("\n")

SECTION #1
SOURCE FILE      : zr_01_rules_of_construction.md
PARENT SECTION   : 12-01
SECTION ID       : 12-01
SECTION TITLE    : Rules Applying to Text of Resolution

CONTENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-01: Rules Applying to Text of Resolution
**Source:** NYC Department of City Planning — zr.planning.nyc.gov  
**Last Amended:** 2/2/2011  
**Retrieved:** March 2026

The following rules of construction apply to the text of this Resolution:



SECTION #2
SOURCE FILE      : zr_02_definitions_key.md
PARENT SECTION   : 12-10 (Excerpt)
SECTION ID       : 12-10 (Excerpt)
SECTION TITLE    : Selected Definitions

CONTENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-10 (Excerpt): Selected Definitions
**Source:** NYC Department of City Planning — zr.planning.nyc.gov  
**Last Amended:** 8/14/2025  
**Retrieved:** March 2026

Words in the text or tables of this Resolution which are italicized shall be in


SECTION #3
SOURCE FILE    

In [8]:
# -------------------------------------------------------------------
# CORPUS STATISTICS
# -------------------------------------------------------------------

total_files = len(raw_markdown_files)

total_sections = len(all_parsed_sections)

empty_sections = []

duplicate_section_ids = set()
seen_section_ids = set()

sections_per_file = {}

for section in all_parsed_sections:
    
    source_file = section["source_file"]
    section_id = section["section_id"]
    content = section["content"].strip()
    
    # ---------------------------------------------------------------
    # COUNT SECTIONS PER FILE
    # ---------------------------------------------------------------
    
    sections_per_file[source_file] = (
        sections_per_file.get(source_file, 0) + 1
    )
    
    # ---------------------------------------------------------------
    # EMPTY CONTENT CHECK
    # ---------------------------------------------------------------
    
    if not content:
        empty_sections.append(section_id)
    
    # ---------------------------------------------------------------
    # DUPLICATE CHECK
    # ---------------------------------------------------------------
    
    if section_id in seen_section_ids:
        duplicate_section_ids.add(section_id)
    
    seen_section_ids.add(section_id)


# -------------------------------------------------------------------
# PRINT STATS
# -------------------------------------------------------------------

print("=" * 80)
print("CORPUS STATISTICS")
print("=" * 80)

print(f"\nTOTAL FILES: {total_files}")

print(f"TOTAL LEGAL SECTIONS: {total_sections}")

print(f"\nEMPTY SECTIONS: {len(empty_sections)}")

print(f"DUPLICATE SECTION IDS: {len(duplicate_section_ids)}")

print("\nSECTIONS PER FILE:\n")

for file_name, count in sections_per_file.items():
    
    print(f"{file_name:<45} {count}")

CORPUS STATISTICS

TOTAL FILES: 10
TOTAL LEGAL SECTIONS: 15

EMPTY SECTIONS: 0
DUPLICATE SECTION IDS: 1

SECTIONS PER FILE:

zr_01_rules_of_construction.md                1
zr_02_definitions_key.md                      1
zr_03_rear_yard_requirements.md               3
zr_04_permitted_obstructions_rear_yard.md     1
zr_05_floor_area_R6_R12_current.md            1
zr_06_floor_area_R6_R10_SUPERSEDED_2019.md    1
zr_07_front_yard_requirements.md              2
zr_08_permitted_obstructions_all_yards.md     1
zr_09_ceqr_e_designations.md                  1
zr_10_height_setback_R6_R12.md                3


In [9]:
# -------------------------------------------------------------------
# METADATA EXTRACTION HELPERS
# -------------------------------------------------------------------

LAST_AMENDED_PATTERN = re.compile(
    r"\*\*Last Amended:\*\*\s*(.+)"
)

APPLICABLE_DISTRICTS_PATTERN = re.compile(
    r"\*\*Applicable Districts:\*\*\s*(.+)"
)

SUPERSEDED_PATTERN = re.compile(
    r"SUPERSEDED",
    re.IGNORECASE
)


# -------------------------------------------------------------------
# EXTRACT LAST AMENDED
# -------------------------------------------------------------------

def extract_last_amended(text):
    
    match = LAST_AMENDED_PATTERN.search(text)
    
    if match:
        return match.group(1).strip()
    
    return "UNKNOWN"


# -------------------------------------------------------------------
# EXTRACT DISTRICT SCOPE
# -------------------------------------------------------------------

def extract_district_scope(text):
    
    match = APPLICABLE_DISTRICTS_PATTERN.search(text)
    
    if match:
        return match.group(1).strip()
    
    return "ALL"


# -------------------------------------------------------------------
# HISTORICAL / SUPERSEDED DETECTION
# -------------------------------------------------------------------

def detect_historical_document(text):
    
    return bool(
        SUPERSEDED_PATTERN.search(text)
    )

In [10]:
# -------------------------------------------------------------------
# REGEX PATTERNS
# -------------------------------------------------------------------

SECTION_HEADER_PATTERN = re.compile(
    r"^##\s+Section\s+([^\n:]+):\s*(.+)$",
    re.MULTILINE
)

SUBSECTION_HEADER_PATTERN = re.compile(
    r"^###\s+([0-9A-Za-z\-\(\)]+):\s*(.+)$",
    re.MULTILINE
)


# -------------------------------------------------------------------
# PARSE SINGLE MARKDOWN FILE
# -------------------------------------------------------------------

def parse_markdown_file(file_record):
    
    source_file = file_record["source_file"]
    text = file_record["text"]
    
    parsed_sections = []
    
    # ---------------------------------------------------------------
    # FILE LEVEL METADATA
    # ---------------------------------------------------------------
    
    last_amended = extract_last_amended(text)
    
    is_historical = detect_historical_document(text)
    
    # ---------------------------------------------------------------
    # MAIN SECTION
    # ---------------------------------------------------------------
    
    main_section_match = SECTION_HEADER_PATTERN.search(text)
    
    if main_section_match:
        
        parent_section_id = main_section_match.group(1).strip()
        parent_section_title = main_section_match.group(2).strip()
        
    else:
        
        parent_section_id = "UNKNOWN"
        parent_section_title = "UNKNOWN"
    
    # ---------------------------------------------------------------
    # SUBSECTIONS
    # ---------------------------------------------------------------
    
    subsection_matches = list(
        SUBSECTION_HEADER_PATTERN.finditer(text)
    )
    
    # ---------------------------------------------------------------
    # CASE 1:
    # HAS SUBSECTIONS
    # ---------------------------------------------------------------
    
    if subsection_matches:
        
        for idx, match in enumerate(subsection_matches):
            
            subsection_id = match.group(1).strip()
            subsection_title = match.group(2).strip()
            
            start_pos = match.end()
            
            if idx < len(subsection_matches) - 1:
                end_pos = subsection_matches[idx + 1].start()
            else:
                end_pos = len(text)
            
            subsection_content = text[start_pos:end_pos].strip()
            
            district_scope = extract_district_scope(
                subsection_content
            )
            
            # -------------------------------------------------------
            # STABLE CHUNK ID
            # -------------------------------------------------------
            
            stable_chunk_id = (
                f"{source_file}::{subsection_id}"
            )
            
            parsed_sections.append({
                
                # ---------------------------------------------------
                # IDENTIFIERS
                # ---------------------------------------------------
                
                "chunk_id": stable_chunk_id,
                
                "source_file": source_file,
                
                "parent_section_id": parent_section_id,
                "parent_section_title": parent_section_title,
                
                "section_id": subsection_id,
                "section_title": subsection_title,
                
                # ---------------------------------------------------
                # CONTENT
                # ---------------------------------------------------
                
                "content": subsection_content,
                
                # ---------------------------------------------------
                # METADATA
                # ---------------------------------------------------
                
                "last_amended": last_amended,
                
                "district_scope": district_scope,
                
                "is_historical": is_historical
            })
    
    # ---------------------------------------------------------------
    # CASE 2:
    # NO SUBSECTIONS
    # ---------------------------------------------------------------
    
    else:
        
        district_scope = extract_district_scope(text)
        
        stable_chunk_id = (
            f"{source_file}::{parent_section_id}"
        )
        
        parsed_sections.append({
            
            "chunk_id": stable_chunk_id,
            
            "source_file": source_file,
            
            "parent_section_id": parent_section_id,
            "parent_section_title": parent_section_title,
            
            "section_id": parent_section_id,
            "section_title": parent_section_title,
            
            "content": text.strip(),
            
            "last_amended": last_amended,
            
            "district_scope": district_scope,
            
            "is_historical": is_historical
        })
    
    return parsed_sections

In [11]:
# -------------------------------------------------------------------
# REBUILD PARSED CORPUS
# -------------------------------------------------------------------

all_parsed_sections = []

for file_record in raw_markdown_files:
    
    parsed_sections = parse_markdown_file(file_record)
    
    all_parsed_sections.extend(parsed_sections)

print("TOTAL PARSED LEGAL SECTIONS:")
print(len(all_parsed_sections))

TOTAL PARSED LEGAL SECTIONS:
15


In [12]:
# -------------------------------------------------------------------
# VALIDATE ENRICHED METADATA
# -------------------------------------------------------------------

for section in all_parsed_sections:
    
    print("=" * 100)
    
    print("CHUNK ID:")
    print(section["chunk_id"])
    
    print("\nSECTION ID:")
    print(section["section_id"])
    
    print("\nTITLE:")
    print(section["section_title"])
    
    print("\nLAST AMENDED:")
    print(section["last_amended"])
    
    print("\nDISTRICT SCOPE:")
    print(section["district_scope"])
    
    print("\nIS HISTORICAL:")
    print(section["is_historical"])
    
    print("\nCONTENT PREVIEW:")
    print(section["content"][:250])
    
    print("\n")

CHUNK ID:
zr_01_rules_of_construction.md::12-01

SECTION ID:
12-01

TITLE:
Rules Applying to Text of Resolution

LAST AMENDED:
2/2/2011

DISTRICT SCOPE:
ALL

IS HISTORICAL:
False

CONTENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-01: Rules Applying to Text of Resolution
**Source:** NYC Department of City Planning — zr.planning.nyc.gov  
**Last Amended:** 2/2/2011  
**Retrieved:** March 2026

The following rules of c


CHUNK ID:
zr_02_definitions_key.md::12-10 (Excerpt)

SECTION ID:
12-10 (Excerpt)

TITLE:
Selected Definitions

LAST AMENDED:
8/14/2025

DISTRICT SCOPE:
ALL

IS HISTORICAL:
False

CONTENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-10 (Excerpt): Selected Definitions
**Source:** NYC Department of City Planning — zr.planning.nyc.gov  
**Last Amended:** 8/14/2025  
**Retrieved:** March 2026

Words in the text or tables o


CHUNK ID:
zr_03_rear_yard_requirements.md::23-342

SECTION ID:
23-342

TITLE:
Rear Yard Requirements


In [13]:
# -------------------------------------------------------------------
# VERIFY CHUNK ID UNIQUENESS
# -------------------------------------------------------------------

chunk_ids = [
    section["chunk_id"]
    for section in all_parsed_sections
]

unique_chunk_ids = set(chunk_ids)

print("TOTAL CHUNK IDS:", len(chunk_ids))

print("UNIQUE CHUNK IDS:", len(unique_chunk_ids))

if len(chunk_ids) == len(unique_chunk_ids):
    
    print("\nSUCCESS: ALL CHUNK IDS ARE UNIQUE")
    
else:
    
    print("\nERROR: DUPLICATE CHUNK IDS DETECTED")

TOTAL CHUNK IDS: 15
UNIQUE CHUNK IDS: 15

SUCCESS: ALL CHUNK IDS ARE UNIQUE


# Stage 2 — Dependency Extraction

In [14]:
# -------------------------------------------------------------------
# CROSS REFERENCE REGEX PATTERNS
# -------------------------------------------------------------------

SECTION_REF_PATTERN = re.compile(
    r"""
    (?:
        Section(?:s)?
        \s+
    )?
    (
        \d{2}-\d{2,4}
        (?:\([a-z0-9]+\))?
    )
    """,
    re.VERBOSE
)

ARTICLE_REF_PATTERN = re.compile(
    r"Article\s+([IVXLC]+)",
    re.IGNORECASE
)

APPENDIX_REF_PATTERN = re.compile(
    r"Appendix\s+([A-Z])",
    re.IGNORECASE
)


# -------------------------------------------------------------------
# EXTRACT LEGAL REFERENCES
# -------------------------------------------------------------------

def extract_legal_references(text):
    
    # ---------------------------------------------------------------
    # SECTION REFERENCES
    # ---------------------------------------------------------------
    
    raw_section_refs = SECTION_REF_PATTERN.findall(text)
    
    section_refs = sorted(
        set(ref.strip() for ref in raw_section_refs)
    )
    
    # ---------------------------------------------------------------
    # ARTICLE REFERENCES
    # ---------------------------------------------------------------
    
    raw_article_refs = ARTICLE_REF_PATTERN.findall(text)
    
    article_refs = sorted(
        set(ref.strip().upper() for ref in raw_article_refs)
    )
    
    # ---------------------------------------------------------------
    # APPENDIX REFERENCES
    # ---------------------------------------------------------------
    
    raw_appendix_refs = APPENDIX_REF_PATTERN.findall(text)
    
    appendix_refs = sorted(
        set(ref.strip().upper() for ref in raw_appendix_refs)
    )
    
    return {
        "section_refs": section_refs,
        "article_refs": article_refs,
        "appendix_refs": appendix_refs
    }

In [15]:
# -------------------------------------------------------------------
# TEST REFERENCE EXTRACTION
# -------------------------------------------------------------------

test_section = all_parsed_sections[1]

print("SECTION:")
print(test_section["section_id"])

print("\nTITLE:")
print(test_section["section_title"])

print("\n" + "="*80 + "\n")

references = extract_legal_references(
    test_section["content"]
)

print(json.dumps(
    references,
    indent=2
))

SECTION:
12-10 (Excerpt)

TITLE:
Selected Definitions


{
  "section_refs": [
    "12-10"
  ],
  "article_refs": [
    "I"
  ],
  "appendix_refs": []
}


In [16]:
# -------------------------------------------------------------------
# ENRICH ALL SECTIONS WITH REFERENCES
# -------------------------------------------------------------------

for section in all_parsed_sections:
    
    extracted_refs = extract_legal_references(
        section["content"]
    )
    
    # ---------------------------------------------------------------
    # REMOVE SELF REFERENCES
    # ---------------------------------------------------------------
    
    current_section_id = section["section_id"]
    
    cleaned_section_refs = [
        ref
        for ref in extracted_refs["section_refs"]
        if ref != current_section_id
    ]
    
    # ---------------------------------------------------------------
    # STORE CLEAN REFERENCES
    # ---------------------------------------------------------------
    
    section["section_refs"] = cleaned_section_refs
    
    section["article_refs"] = (
        extracted_refs["article_refs"]
    )
    
    section["appendix_refs"] = (
        extracted_refs["appendix_refs"]
    )

print("REFERENCE EXTRACTION COMPLETE")

REFERENCE EXTRACTION COMPLETE


In [17]:
# -------------------------------------------------------------------
# INSPECT DEPENDENCY EXTRACTION
# -------------------------------------------------------------------

for section in all_parsed_sections:
    
    print("=" * 100)
    
    print("SECTION:")
    print(section["section_id"])
    
    print("\nTITLE:")
    print(section["section_title"])
    
    print("\nSECTION REFERENCES:")
    print(section["section_refs"])
    
    print("\nARTICLE REFERENCES:")
    print(section["article_refs"])
    
    print("\nAPPENDIX REFERENCES:")
    print(section["appendix_refs"])
    
    print("\n")

SECTION:
12-01

TITLE:
Rules Applying to Text of Resolution

SECTION REFERENCES:
['12-02']

ARTICLE REFERENCES:
['I', 'II', 'III', 'IV']

APPENDIX REFERENCES:
[]


SECTION:
12-10 (Excerpt)

TITLE:
Selected Definitions

SECTION REFERENCES:
['12-10']

ARTICLE REFERENCES:
['I']

APPENDIX REFERENCES:
[]


SECTION:
23-342

TITLE:
Rear Yard Requirements

SECTION REFERENCES:
['23-341', '23-344']

ARTICLE REFERENCES:
[]

APPENDIX REFERENCES:
[]


SECTION:
23-343

TITLE:
Rear Yard Equivalent Requirements

SECTION REFERENCES:
['23-342']

ARTICLE REFERENCES:
[]

APPENDIX REFERENCES:
[]


SECTION:
23-344

TITLE:
Additional Rear Yard Modifications

SECTION REFERENCES:
['23-711']

ARTICLE REFERENCES:
['VII']

APPENDIX REFERENCES:
[]


SECTION:
23-341

TITLE:
Permitted Obstructions in Required Rear Yards or Rear Yard Equivalents

SECTION REFERENCES:
['12-01(a)', '26-50']

ARTICLE REFERENCES:
['II']

APPENDIX REFERENCES:
[]


SECTION:
23-22

TITLE:
Floor Area Regulations for R6 Through R12 Districts



In [18]:
# -------------------------------------------------------------------
# BUILD VERSION-SAFE LEGAL GRAPH
# -------------------------------------------------------------------

legal_dependency_graph = {}

for section in all_parsed_sections:
    
    chunk_id = section["chunk_id"]
    
    legal_dependency_graph[chunk_id] = {
        
        "section_id": section["section_id"],
        
        "source_file": section["source_file"],
        
        "section_title": section["section_title"],
        
        "section_refs": section["section_refs"],
        
        "article_refs": section["article_refs"],
        
        "appendix_refs": section["appendix_refs"],
        
        "is_historical": section["is_historical"]
    }

print("DEPENDENCY GRAPH NODES:")
print(len(legal_dependency_graph))

DEPENDENCY GRAPH NODES:
15


In [19]:
# -------------------------------------------------------------------
# EXPORT SECTION MAP
# -------------------------------------------------------------------

output_path = Path("./section_map.json")

with output_path.open(
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        legal_dependency_graph,
        f,
        indent=2,
        ensure_ascii=False
    )

print("SECTION MAP SAVED:")
print(output_path.resolve())

SECTION MAP SAVED:
D:\sk\planso_assignment\section_map.json


# Stage 3 — Production Chunk Objects + Chroma Metadata

In [20]:
# -------------------------------------------------------------------
# CHROMA SAFE SERIALIZATION HELPERS
# -------------------------------------------------------------------

def serialize_reference_list(ref_list):
    
    """
    Convert list references into
    Chroma-safe pipe-separated strings.
    """
    
    if not ref_list:
        return "NONE"
    
    return "|".join(sorted(set(ref_list)))


def boolean_to_string(value):
    
    return "true" if value else "false"

In [21]:
# -------------------------------------------------------------------
# BUILD FINAL PRODUCTION CHUNK OBJECTS
# -------------------------------------------------------------------

production_chunks = []

for chunk_index, section in enumerate(all_parsed_sections):
    
    # ---------------------------------------------------------------
    # SERIALIZED REFERENCES
    # ---------------------------------------------------------------
    
    serialized_section_refs = serialize_reference_list(
        section["section_refs"]
    )
    
    serialized_article_refs = serialize_reference_list(
        section["article_refs"]
    )
    
    serialized_appendix_refs = serialize_reference_list(
        section["appendix_refs"]
    )
    
    # ---------------------------------------------------------------
    # CHROMA SAFE METADATA
    # ---------------------------------------------------------------
    
    chroma_metadata = {
        
        # -----------------------------------------------------------
        # CORE IDENTIFIERS
        # -----------------------------------------------------------
        
        "chunk_id": section["chunk_id"],
        
        "section_id": section["section_id"],
        
        "section_title": section["section_title"],
        
        "parent_section_id": section["parent_section_id"],
        
        "source_file": section["source_file"],
        
        # -----------------------------------------------------------
        # LEGAL METADATA
        # -----------------------------------------------------------
        
        "last_amended": section["last_amended"],
        
        "district_scope": section["district_scope"],
        
        "is_historical": boolean_to_string(
            section["is_historical"]
        ),
        
        # -----------------------------------------------------------
        # LEGAL DEPENDENCIES
        # -----------------------------------------------------------
        
        "cross_refs": serialized_section_refs,
        
        "article_refs": serialized_article_refs,
        
        "appendix_refs": serialized_appendix_refs,
        
        # -----------------------------------------------------------
        # TRACEABILITY
        # -----------------------------------------------------------
        
        "chunk_index": chunk_index
    }
    
    # ---------------------------------------------------------------
    # FINAL PRODUCTION OBJECT
    # ---------------------------------------------------------------
    
    production_chunk = {
        
        "id": section["chunk_id"],
        
        "document": section["content"],
        
        "metadata": chroma_metadata
    }
    
    production_chunks.append(
        production_chunk
    )

print("PRODUCTION CHUNKS CREATED:")
print(len(production_chunks))

PRODUCTION CHUNKS CREATED:
15


In [22]:
# -------------------------------------------------------------------
# INSPECT PRODUCTION CHUNKS
# -------------------------------------------------------------------

for chunk in production_chunks[:5]:
    
    print("=" * 100)
    
    print("ID:")
    print(chunk["id"])
    
    print("\nDOCUMENT PREVIEW:")
    print(chunk["document"][:300])
    
    print("\nMETADATA:")
    
    pprint(chunk["metadata"])
    
    print("\n")

ID:
zr_01_rules_of_construction.md::12-01

DOCUMENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-01: Rules Applying to Text of Resolution
**Source:** NYC Department of City Planning — zr.planning.nyc.gov  
**Last Amended:** 2/2/2011  
**Retrieved:** March 2026

The following rules of construction apply to the text of this Resolution:


METADATA:
{'appendix_refs': 'NONE',
 'article_refs': 'I|II|III|IV',
 'chunk_id': 'zr_01_rules_of_construction.md::12-01',
 'chunk_index': 0,
 'cross_refs': '12-02',
 'district_scope': 'ALL',
 'is_historical': 'false',
 'last_amended': '2/2/2011',
 'parent_section_id': '12-01',
 'section_id': '12-01',
 'section_title': 'Rules Applying to Text of Resolution',
 'source_file': 'zr_01_rules_of_construction.md'}


ID:
zr_02_definitions_key.md::12-10 (Excerpt)

DOCUMENT PREVIEW:
# NYC Zoning Resolution — Article I, Chapter 2
## Section 12-10 (Excerpt): Selected Definitions
**Source:** NYC Department of City Planning — zr.planning.nyc.go

In [23]:
# -------------------------------------------------------------------
# VALIDATE CHROMA METADATA TYPES
# -------------------------------------------------------------------

allowed_types = (
    str,
    int,
    float,
    bool
)

invalid_metadata_entries = []

for chunk in production_chunks:
    
    metadata = chunk["metadata"]
    
    for key, value in metadata.items():
        
        if not isinstance(value, allowed_types):
            
            invalid_metadata_entries.append({
                "chunk_id": chunk["id"],
                "metadata_key": key,
                "invalid_type": type(value)
            })

print("=" * 80)

print("INVALID METADATA ENTRIES:")
print(len(invalid_metadata_entries))

if invalid_metadata_entries:
    
    pprint(invalid_metadata_entries[:5])

else:
    
    print("\nSUCCESS: ALL METADATA IS CHROMA SAFE")

INVALID METADATA ENTRIES:
0

SUCCESS: ALL METADATA IS CHROMA SAFE


In [24]:
# -------------------------------------------------------------------
# RETRIEVAL CORPUS STATISTICS
# -------------------------------------------------------------------

historical_chunks = 0

chunks_with_cross_refs = 0

total_cross_refs = 0

district_scopes = set()

for chunk in production_chunks:
    
    metadata = chunk["metadata"]
    
    # ---------------------------------------------------------------
    # HISTORICAL
    # ---------------------------------------------------------------
    
    if metadata["is_historical"] == "true":
        historical_chunks += 1
    
    # ---------------------------------------------------------------
    # CROSS REFERENCES
    # ---------------------------------------------------------------
    
    if metadata["cross_refs"] != "NONE":
        
        chunks_with_cross_refs += 1
        
        total_cross_refs += len(
            metadata["cross_refs"].split("|")
        )
    
    # ---------------------------------------------------------------
    # DISTRICTS
    # ---------------------------------------------------------------
    
    district_scopes.add(
        metadata["district_scope"]
    )

print("=" * 80)

print("RETRIEVAL CORPUS STATISTICS")

print("=" * 80)

print(f"\nTOTAL CHUNKS: {len(production_chunks)}")

print(f"\nHISTORICAL CHUNKS: {historical_chunks}")

print(f"\nCHUNKS WITH CROSS REFERENCES: {chunks_with_cross_refs}")

print(f"\nTOTAL EXTRACTED CROSS REFERENCES: {total_cross_refs}")

print(f"\nUNIQUE DISTRICT SCOPES: {len(district_scopes)}")

RETRIEVAL CORPUS STATISTICS

TOTAL CHUNKS: 15

HISTORICAL CHUNKS: 1

CHUNKS WITH CROSS REFERENCES: 12

TOTAL EXTRACTED CROSS REFERENCES: 21

UNIQUE DISTRICT SCOPES: 2


In [25]:
# -------------------------------------------------------------------
# BUILD CHROMA INGESTION ARRAYS
# -------------------------------------------------------------------

chroma_ids = []

chroma_documents = []

chroma_metadatas = []

for chunk in production_chunks:
    
    chroma_ids.append(
        chunk["id"]
    )
    
    chroma_documents.append(
        chunk["document"]
    )
    
    chroma_metadatas.append(
        chunk["metadata"]
    )

print("CHROMA INGESTION ARRAYS READY")

print(f"\nIDS: {len(chroma_ids)}")

print(f"DOCUMENTS: {len(chroma_documents)}")

print(f"METADATAS: {len(chroma_metadatas)}")

CHROMA INGESTION ARRAYS READY

IDS: 15
DOCUMENTS: 15
METADATAS: 15


In [26]:
# -------------------------------------------------------------------
# FINAL INGESTION VALIDATION
# -------------------------------------------------------------------

assert len(chroma_ids) == len(chroma_documents)

assert len(chroma_documents) == len(chroma_metadatas)

assert len(set(chroma_ids)) == len(chroma_ids)

print("SUCCESS: INGESTION VALIDATION PASSED")

SUCCESS: INGESTION VALIDATION PASSED


In [27]:
# -------------------------------------------------------------------
# EXPORT PRODUCTION CHUNKS
# -------------------------------------------------------------------

output_path = Path("./processed_production_chunks.json")

with output_path.open(
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        production_chunks,
        f,
        indent=2,
        ensure_ascii=False
    )

print("PRODUCTION CHUNKS SAVED")

print(output_path.resolve())

PRODUCTION CHUNKS SAVED
D:\sk\planso_assignment\processed_production_chunks.json


# Stage 4

In [28]:
!pip install chromadb sentence-transformers -q

In [29]:
# -------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------

import chromadb

from chromadb.config import Settings

from sentence_transformers import SentenceTransformer

import numpy as np

In [30]:
# -------------------------------------------------------------------
# LOAD EMBEDDING MODEL
# -------------------------------------------------------------------

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded:")
print(EMBEDDING_MODEL_NAME)

Embedding model loaded:
all-MiniLM-L6-v2


In [31]:
# -------------------------------------------------------------------
# CHROMADB STORAGE PATH
# -------------------------------------------------------------------

CHROMA_DB_PATH = "./chroma_legal_db"


# -------------------------------------------------------------------
# INITIALIZE PERSISTENT CLIENT
# -------------------------------------------------------------------

chroma_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

print("Persistent ChromaDB initialized")

print("\nDB Path:")
print(Path(CHROMA_DB_PATH).resolve())

Persistent ChromaDB initialized

DB Path:
D:\sk\planso_assignment\chroma_legal_db


In [32]:
# -------------------------------------------------------------------
# COLLECTION NAME
# -------------------------------------------------------------------

COLLECTION_NAME = "nyc_zoning_resolution"


# -------------------------------------------------------------------
# DELETE OLD COLLECTION IF EXISTS
# -------------------------------------------------------------------

existing_collections = chroma_client.list_collections()

existing_names = [
    c.name for c in existing_collections
]

if COLLECTION_NAME in existing_names:
    
    chroma_client.delete_collection(
        COLLECTION_NAME
    )
    
    print("Old collection deleted")


# -------------------------------------------------------------------
# CREATE NEW COLLECTION
# -------------------------------------------------------------------

legal_collection = chroma_client.create_collection(
    
    name=COLLECTION_NAME,
    
    metadata={
        "description": (
            "NYC zoning resolution legal corpus"
        )
    }
)

print("\nFresh collection created:")
print(COLLECTION_NAME)


Fresh collection created:
nyc_zoning_resolution


In [33]:
# -------------------------------------------------------------------
# GENERATE EMBEDDINGS
# -------------------------------------------------------------------

print("Generating embeddings...")

document_embeddings = embedding_model.encode(
    chroma_documents,
    show_progress_bar=True
)

print("\nEmbeddings generated:")
print(len(document_embeddings))


# -------------------------------------------------------------------
# INGEST INTO CHROMADB
# -------------------------------------------------------------------

legal_collection.add(
    
    ids=chroma_ids,
    
    documents=chroma_documents,
    
    metadatas=chroma_metadatas,
    
    embeddings=document_embeddings.tolist()
)

print("\nDocuments ingested into ChromaDB")

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embeddings generated:
15

Documents ingested into ChromaDB


In [34]:
# -------------------------------------------------------------------
# VERIFY COLLECTION
# -------------------------------------------------------------------

collection_count = legal_collection.count()

print("TOTAL DOCUMENTS IN COLLECTION:")
print(collection_count)

TOTAL DOCUMENTS IN COLLECTION:
15


In [35]:
# -------------------------------------------------------------------
# SEMANTIC RETRIEVAL
# -------------------------------------------------------------------

SIMILARITY_THRESHOLD = 0.38


def semantic_search(
    query,
    top_k=5
):
    
    # ---------------------------------------------------------------
    # EMBED QUERY
    # ---------------------------------------------------------------
    
    query_embedding = embedding_model.encode(
        query
    ).tolist()
    
    # ---------------------------------------------------------------
    # CHROMA QUERY
    # ---------------------------------------------------------------
    
    results = legal_collection.query(
        
        query_embeddings=[query_embedding],
        
        n_results=top_k
    )
    
    retrieved_chunks = []
    
    documents = results["documents"][0]
    
    metadatas = results["metadatas"][0]
    
    distances = results["distances"][0]
    
    ids = results["ids"][0]
    
    # ---------------------------------------------------------------
    # FILTER RESULTS
    # ---------------------------------------------------------------
    
    for doc, meta, distance, chunk_id in zip(
        documents,
        metadatas,
        distances,
        ids
    ):
        
        similarity_score = 1 - distance
        
        if similarity_score >= SIMILARITY_THRESHOLD:
            
            retrieved_chunks.append({
                
                "chunk_id": chunk_id,
                
                "score": round(similarity_score, 4),
                
                "document": doc,
                
                "metadata": meta
            })
    
    return retrieved_chunks

In [36]:
# -------------------------------------------------------------------
# TEST SEMANTIC RETRIEVAL
# -------------------------------------------------------------------

query = "rear yard obstruction allowances"

results = semantic_search(
    query=query,
    top_k=5
)

print("QUERY:")
print(query)

print("\n" + "="*100)

print(f"\nRESULTS FOUND: {len(results)}\n")

for result in results:
    
    print("=" * 100)
    
    print("CHUNK ID:")
    print(result["chunk_id"])
    
    print("\nSECTION:")
    print(result["metadata"]["section_id"])
    
    print("\nTITLE:")
    print(result["metadata"]["section_title"])
    
    print("\nSIMILARITY:")
    print(result["score"])
    
    print("\nCROSS REFS:")
    print(result["metadata"]["cross_refs"])
    
    print("\nDOCUMENT PREVIEW:")
    print(result["document"][:400])
    
    print("\n")

QUERY:
rear yard obstruction allowances


RESULTS FOUND: 1

CHUNK ID:
zr_03_rear_yard_requirements.md::23-342

SECTION:
23-342

TITLE:
Rear Yard Requirements

SIMILARITY:
0.4062

CROSS REFS:
23-341|23-344

DOCUMENT PREVIEW:
**Applicable Districts:** R1 through R10

In all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).

The minimum required **rear yard** depth shall be:

| District |




In [37]:
# -------------------------------------------------------------------
# ONE HOP DEPENDENCY EXPANSION
# -------------------------------------------------------------------

def expand_dependencies(
    retrieved_chunks
):
    
    expanded_chunks = []
    
    seen_chunk_ids = set()
    
    # ---------------------------------------------------------------
    # TRACK ORIGINAL RESULTS
    # ---------------------------------------------------------------
    
    for chunk in retrieved_chunks:
        
        seen_chunk_ids.add(
            chunk["chunk_id"]
        )
        
        expanded_chunks.append(chunk)
    
    # ---------------------------------------------------------------
    # DEPENDENCY EXPANSION
    # ---------------------------------------------------------------
    
    for chunk in retrieved_chunks:
        
        cross_refs = chunk["metadata"]["cross_refs"]
        
        if cross_refs == "NONE":
            continue
        
        referenced_sections = cross_refs.split("|")
        
        for ref_section_id in referenced_sections:
            
            # -------------------------------------------------------
            # FIND MATCHING CHUNKS
            # -------------------------------------------------------
            
            for candidate in production_chunks:
                
                candidate_section_id = (
                    candidate["metadata"]["section_id"]
                )
                
                candidate_chunk_id = (
                    candidate["id"]
                )
                
                if (
                    candidate_section_id == ref_section_id
                    and
                    candidate_chunk_id not in seen_chunk_ids
                ):
                    
                    expanded_chunks.append({
                        
                        "chunk_id": candidate_chunk_id,
                        
                        "score": "DEPENDENCY",
                        
                        "document": candidate["document"],
                        
                        "metadata": candidate["metadata"]
                    })
                    
                    seen_chunk_ids.add(
                        candidate_chunk_id
                    )
    
    return expanded_chunks

In [38]:
# -------------------------------------------------------------------
# FINAL LEGAL RETRIEVAL PIPELINE
# -------------------------------------------------------------------

query = "rear yard obstruction allowances"

# ---------------------------------------------------------------
# STEP 1: SEMANTIC SEARCH
# ---------------------------------------------------------------

semantic_results = semantic_search(
    query=query,
    top_k=5
)

# ---------------------------------------------------------------
# STEP 2: DEPENDENCY EXPANSION
# ---------------------------------------------------------------

final_results = expand_dependencies(
    semantic_results
)

print("=" * 100)

print("FINAL DEPENDENCY-AWARE RESULTS")

print("=" * 100)

print(f"\nTOTAL RESULTS: {len(final_results)}\n")

for result in final_results:
    
    print("=" * 100)
    
    print("SECTION:")
    print(result["metadata"]["section_id"])
    
    print("\nTITLE:")
    print(result["metadata"]["section_title"])
    
    print("\nSOURCE:")
    print(result["metadata"]["source_file"])
    
    print("\nRETRIEVAL TYPE:")
    print(result["score"])
    
    print("\nCROSS REFERENCES:")
    print(result["metadata"]["cross_refs"])
    
    print("\nDOCUMENT PREVIEW:")
    print(result["document"][:400])
    
    print("\n")

FINAL DEPENDENCY-AWARE RESULTS

TOTAL RESULTS: 3

SECTION:
23-342

TITLE:
Rear Yard Requirements

SOURCE:
zr_03_rear_yard_requirements.md

RETRIEVAL TYPE:
0.4062

CROSS REFERENCES:
23-341|23-344

DOCUMENT PREVIEW:
**Applicable Districts:** R1 through R10

In all districts, as indicated, a **rear yard** shall be provided at every required **rear lot line** of a **zoning lot**, except as otherwise provided in Sections 23-341 (Permitted obstructions in required rear yards or rear yard equivalents) and 23-344 (Additional rear yard modifications).

The minimum required **rear yard** depth shall be:

| District |


SECTION:
23-341

TITLE:
Permitted Obstructions in Required Rear Yards or Rear Yard Equivalents

SOURCE:
zr_04_permitted_obstructions_rear_yard.md

RETRIEVAL TYPE:
DEPENDENCY

CROSS REFERENCES:
12-01(a)|26-50

DOCUMENT PREVIEW:
# NYC Zoning Resolution — Article II, Chapter 3
## Section 23-341: Permitted Obstructions in Required Rear Yards or Rear Yard Equivalents
**Source:** NYC De